In [1]:
# 1. Comandos Mágicos de Jupyter para recargar módulos locales automáticamente
%load_ext autoreload
%autoreload 2

# 2. Tu código para encontrar la carpeta src/
import sys
sys.path.append('..')

# 3. El resto de importaciones normales
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config

from src.transformers import (
    DropColumnsTransformer,
    UnknownToNaNTransformer,
    DropHighMissingTransformer,
    OutlierCapper,
    DropZeroVarianceTransformer,
    SmartImputerTransformer
)

print("✅ Librerías locales importadas correctamente.")

✅ Librerías locales importadas correctamente.


In [2]:
# Cargamos los datos crudos
df_raw = pd.read_csv('../data/raw/bank-full.csv', sep=';')

# Definimos qué columnas van por qué ruta
variables_numericas = ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
variables_categoricas = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month']


# 1. Ruta para números (Capping con interruptor, Varianza Cero, Escalar)
num_pipe = Pipeline([
    ('capper', OutlierCapper(apply_capping=True)), # ¡Nuestro nuevo switch!
    ('zero_variance', DropZeroVarianceTransformer()), # ¡Limpieza matemática!
    ('scaler', StandardScaler())                     
])

# 2. Ruta para textos (Solo codificar, ya que SmartImputer imputó antes)
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)) 
])

# 3. El enrutador
preprocesador = ColumnTransformer(
    transformers=[
        ('num', num_pipe, variables_numericas),
        ('cat', cat_pipe, variables_categoricas)
    ], remainder='drop'
)

# 4. EL SÚPER PIPELINE
pipeline_ml_completo = Pipeline([
    ('drop_leaks', DropColumnsTransformer(columns_to_drop=['duration'])),
    ('limpieza_unknowns', UnknownToNaNTransformer()),
    ('drop_high_nan', DropHighMissingTransformer(threshold=0.8)), 
    ('smart_imputer', SmartImputerTransformer(low_threshold=0.10)), # El imputador general
    ('preprocesamiento', preprocesador)
])

set_config(display="diagram")
pipeline_ml_completo

Pipeline(steps=[('drop_leaks',
                 DropColumnsTransformer(columns_to_drop=['duration'])),
                ('limpieza_unknowns', UnknownToNaNTransformer()),
                ('drop_high_nan', DropHighMissingTransformer()),
                ('smart_imputer', SmartImputerTransformer()),
                ('preprocesamiento',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('capper',
                                                                   OutlierCapper()),
                                                                  ('zero_variance',
                                                                   DropZeroVarianceTransformer()),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['age', 'balance', 'day',
                                                   'campaign', 'pdays',
                                                   'previous']),
                                                 ('cat',
                                                  Pipeline(steps=[('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['job', 'marital',
                                                   'education', 'default',
                                                   'housing', 'loan', 'contact',
                                                   'month'])]))])

In [3]:
# ¡EL MOMENTO DE LA VERDAD!
print("Dimensiones originales:", df_raw.shape)
df_procesado = pipeline_ml_completo.fit_transform(df_raw)
print("Dimensiones procesadas:", df_procesado.shape)

print(f"\nTipo de dato de salida: {type(df_procesado)}")
print("\nPrimeras 3 filas de la matriz matemática lista para el modelo:")
print(df_procesado[:3])

Dimensiones originales: (45211, 17)
🧠 SmartImputer - Simples (<10%): ['job', 'education']
🚧 SmartImputer - Complejas (>10%): ['contact'] (PENDIENTE)
Dimensiones procesadas: (45211, 41)

Tipo de dato de salida: <class 'numpy.ndarray'>

Primeras 3 filas de la matriz matemática lista para el modelo:
[[ 1.64811715  1.02765295 -1.29847633 -0.8700916   0.          0.
   0.          0.          1.          0.          0.          0.
   0.          0.          0.          0.          1.          0.
   0.          0.          1.          1.          0.          0.
   1.          1.          0.          1.          0.          0.
   0.          0.          0.          0.          0.          0.
   0.          1.          0.          0.          0.        ]
 [ 0.30128731 -0.7688168  -1.29847633 -0.8700916   0.          0.
   0.          0.          0.          0.          0.          0.
   0.          1.          0.          0.          0.          1.
   0.          1.          0.          1.    

In [4]:
# Accedemos al paso del 'preprocesamiento' (el ColumnTransformer)
enrutador = pipeline_ml_completo.named_steps['preprocesamiento']

# Le pedimos que nos devuelva los nombres que generó
nombres_finales = enrutador.get_feature_names_out()

print(f"Total de variables finales listas para el modelo: {len(nombres_finales)}\n")

# Imprimimos la lista numerada
for i, nombre in enumerate(nombres_finales):
    # Limpiamos el nombre para que se lea mejor (quitando el prefijo 'num__' o 'cat__')
    nombre_limpio = nombre.replace('num__', '').replace('cat__', '')
    print(f"{i+1:02d}. {nombre_limpio}")

Total de variables finales listas para el modelo: 41

01. age
02. balance
03. day
04. campaign
05. job_admin.
06. job_blue-collar
07. job_entrepreneur
08. job_housemaid
09. job_management
10. job_retired
11. job_self-employed
12. job_services
13. job_student
14. job_technician
15. job_unemployed
16. marital_divorced
17. marital_married
18. marital_single
19. education_primary
20. education_secondary
21. education_tertiary
22. default_no
23. default_yes
24. housing_no
25. housing_yes
26. loan_no
27. loan_yes
28. contact_cellular
29. contact_telephone
30. month_apr
31. month_aug
32. month_dec
33. month_feb
34. month_jan
35. month_jul
36. month_jun
37. month_mar
38. month_may
39. month_nov
40. month_oct
41. month_sep
